# Columnwise-K vs. Brute Force

We want to choose a 4-bit format that keeps the multiplication result close to the original result. Columnwise-K checks selected columns. The full methods check the complete matrix.


## Settings

Set the input paths and the percentage of columns to keep. The remaining settings define the five formats used by both methods.


In [1]:
from pathlib import Path
import math
import time
import itertools
import numpy as np
import pandas as pd
from IPython.display import display

DATA_NPZ_PATH  =  None
W_PATH  =  "c:/Users/Admin/OneDrive/Desktop/System/W_C.npy"
X_CAL_PATH  =  "c:/Users/Admin/OneDrive/Desktop/System/X_cal.npy"

K_COLUMN_PCT  =  27.5

E8M0_MIN_EXP  =   - 127
E8M0_MAX_EXP  =  127
BFP_EXP_POLICY  =  "autoflex_predict"
BFP_ALPHA  =  1.0
BFP_SHARING  =  "block"
BFP_BLOCK_SIZE  =  32

FORMAT_CONFIGS  =  {
    "E2M1": {
        "kind": "mxfp",  "block_size": 32,  "element_bits": 4,
        "metadata_bits": 8,  "layout": "[S|E|M] = [1|2|1]",
        "grid": np.array([0.0,  0.5,  1.0,  1.5,  2.0,  3.0,  4.0,  6.0],  dtype = np.float64),
        "baseline_exp_shift":  - 2,  "exp_min": E8M0_MIN_EXP,  "exp_max": E8M0_MAX_EXP,
        "shared_scale": "power-of-two 2^e; 8-bit E8M0-style shared exponent",
        "convention_status": "OCP MXFP4 E2M1 codebook; shared-scale policy is declared here",
    },
    "E1M2": {
        "kind": "mxfp",  "block_size": 32,  "element_bits": 4,
        "metadata_bits": 8,  "layout": "[S|E|M] = [1|1|2]",
        "grid": np.array([0.0,  0.5,  1.0,  1.5,  2.0,  2.5,  3.0,  3.5],  dtype = np.float64),
        "baseline_exp_shift":  - 1,  "exp_min": E8M0_MIN_EXP,  "exp_max": E8M0_MAX_EXP,
        "shared_scale": "power-of-two 2^e; 8-bit E8M0-style shared exponent",
        "convention_status": "FP4 E1M2 research codebook, used as a project format",
    },
    "MXINT4": {
        "kind": "mxint",  "block_size": 32,  "element_bits": 4,
        "metadata_bits": 8,  "layout": "signed integer codes -8..7",
        "qmin":  - 8,  "qmax": 7,  "element_scale": 0.25,
        "exp_min": E8M0_MIN_EXP,  "exp_max": E8M0_MAX_EXP,
        "shared_scale": "power-of-two 2^e; element precision scale 2^-2",
        "convention_status": "assignment/classroom MXINT4 convention",
    },
    "BFP4": {
        "kind": "bfp",  "block_size": BFP_BLOCK_SIZE,  "element_bits": 4,
        "metadata_bits": 8,  "layout": "Flexpoint-style 4-bit two's-complement mantissa -8..7",
        "mantissa_bits": 4,  "qmin":  - 8,  "qmax": 7,
        "exp_min": E8M0_MIN_EXP,  "exp_max": E8M0_MAX_EXP,
        "shared_scale": "value = m * 2^E; 8-bit shared exponent per 32-weight block",
        "convention_status": "Flexpoint-inspired BFP4 project convention",
    },
    "MXUINT4": {
        "kind": "mxuint",  "block_size": 32,  "element_bits": 4,
        "metadata_bits": 8,  "layout": "unsigned codes 0..15 with fixed zero point Z=8",
        "qmin": 0,  "qmax": 15,  "zero_point": 8,
        "exp_min": E8M0_MIN_EXP,  "exp_max": E8M0_MAX_EXP,
        "shared_scale": "power-of-two 2^e; fixed midpoint zero point 8; no per-block offset",
        "convention_status": "custom midpoint-coded unsigned-storage variant (not an OCP-standard MXUINT4)",
    },
}
FORMAT_ORDER  =  ["E2M1",  "E1M2",  "MXUINT4",  "MXINT4",  "BFP4"]


## Load data and prepare quantization

The next cell loads the weights and calibration inputs. It also defines how each format rounds weights. Nearby weights share a scale, which sets the size of the allowed steps.


In [2]:
def load_inputs():
    if DATA_NPZ_PATH is not None and Path(DATA_NPZ_PATH).is_file():
        with np.load(DATA_NPZ_PATH,  allow_pickle = False) as archive:
            missing  =  [key for key in ("W_C",  "X_cal") if key not in archive.files]

            if missing:
                raise KeyError(f"NPZ archive is missing required keys: {missing}")

            W_input  =  archive["W_C"]
            X_input  =  archive["X_cal"]
            W_source  =  f"{DATA_NPZ_PATH}::W_C"
            X_source  =  f"{DATA_NPZ_PATH}::X_cal"
    else:
        if not Path(W_PATH).is_file():
            raise FileNotFoundError(f"Weight file not found: {W_PATH}. Edit W_PATH in the configuration cell.")

        if not Path(X_CAL_PATH).is_file():
            raise FileNotFoundError(f"Calibration file not found: {X_CAL_PATH}. Edit X_CAL_PATH in the configuration cell.")

        W_input  =  np.load(W_PATH,  allow_pickle = False)
        X_input  =  np.load(X_CAL_PATH,  allow_pickle = False)
        W_source,  X_source  =  str(W_PATH),  str(X_CAL_PATH)

    for name,  a in [("W",  W_input),  ("X_cal",  X_input)]:
        if not isinstance(a,  np.ndarray) or a.ndim  !=  2 or min(a.shape)  ==  0:
            raise ValueError(f"{name} must be a nonempty 2-D array.")

        if not np.issubdtype(a.dtype,  np.number) or np.iscomplexobj(a):
            raise ValueError(f"{name} must contain real numeric values.")

        if not np.isfinite(a).all():
            raise ValueError(f"{name} contains NaN or Inf.")

    if W_input.shape[1]  !=  X_input.shape[0]:
        raise ValueError(f"Shape mismatch: W={W_input.shape}, X_cal={X_input.shape}; need W.shape[1] == X_cal.shape[0].")

    W  =  np.asarray(W_input,  dtype = np.float64)
    X  =  np.asarray(X_input,  dtype = np.float64)
    metadata  =  pd.DataFrame([{
        "Dataset": "C",
        "W_shape": str(W.shape),
        "X_cal_shape": str(X.shape),
        "W_original_dtype": str(W_input.dtype),
        "X_cal_original_dtype": str(X_input.dtype),
        "W_source": W_source,
        "X_cal_source": X_source,
        "Preprocessing": "none (float64 conversion for analysis only)",
    }])
    return W,  X,  metadata

def rowwise_ranges(ncols,  block_size):
    return [(start,  min(start  +  block_size,  ncols)) for start in range(0,  ncols,  block_size)]

def storage_for_format(fmt,  W_shape):
    cfg  =  FORMAT_CONFIGS[fmt]
    nweights  =  int(np.prod(W_shape))

    if fmt  ==  "BFP4" and BFP_SHARING  ==  "tensor":
        nblocks  =  1
    else:
        nblocks  =  W_shape[0]  *  math.ceil(W_shape[1]  /  cfg["block_size"])

    element_bits  =  nweights  *  cfg["element_bits"]
    metadata_bits  =  nblocks  *  cfg["metadata_bits"]
    total_bits  =  element_bits  +  metadata_bits
    return {
        "Blocks": nblocks,
        "Element_bits_total": element_bits,
        "Metadata_bits_total": metadata_bits,
        "Total_storage_bits": total_bits,
        "Effective_bits_per_weight": total_bits  /  nweights,
    }

def nearest_grid_rne(values,  grid):
    """Nearest finite magnitude code; ties choose an even code index."""
    values  =  np.asarray(values,  dtype = np.float64)
    flat  =  values.reshape( - 1)
    out  =  np.empty_like(flat)

    for i,  x in enumerate(flat):
        distances  =  np.abs(grid  -  x)
        dmin  =  np.min(distances)
        candidates  =  np.where(np.isclose(distances,  dmin,  rtol = 0.0,  atol = 1e-15))[0]

        if len(candidates)  ==  1:
            idx  =  int(candidates[0])
        else:
            even  =  candidates[candidates % 2  ==  0]
            idx  =  int(even[0] if len(even) else candidates[0])

        out[i]  =  grid[idx]

    return out.reshape(values.shape)

def bfp_exp_autoflex_predict(block):
    """Choose a scale using the block maximum."""
    nbits  =  FORMAT_CONFIGS["BFP4"]["mantissa_bits"]
    chi  =  BFP_ALPHA  *  float(np.max(np.abs(block)))

    if chi  ==  0:
        return 0

    return int(np.ceil(np.log2(chi)))  -  nbits  +  1

def bfp_exp_autoflex_init(block):
    """Literal Flexpoint-style trial-and-error exponent initialization."""
    nbits  =  FORMAT_CONFIGS["BFP4"]["mantissa_bits"]

    if float(np.max(np.abs(block)))  ==  0:
        return 0

    exponent  =  0

    for _ in range(1024):
        scale  =  np.exp2(exponent)
        gamma  =  float(min(np.max(np.rint(np.abs(np.asarray(block,  dtype = np.float64))  /  scale)),  2  **  (nbits  -  1)  -  1))

        if gamma  >=  2  **  (nbits  -  1)  -  1:
            exponent  +=  (nbits  -  1) // 2
        elif gamma  <  2  **  (nbits  -  2):
            exponent  +=  int(np.ceil(np.log2(max(gamma,  1.0))))  -  (nbits  -  2)

            if gamma  >  2.0  **  ((nbits  -  1) // 2  -  2):
                break
        else:
            break

        exponent  =  int(np.clip(exponent,  E8M0_MIN_EXP,  E8M0_MAX_EXP))

    return exponent

def baseline_exp(block,  fmt):
    """Choose the baseline exponent for this format."""
    block  =  np.asarray(block,  dtype = np.float64)
    amax  =  float(np.max(np.abs(block)))

    if amax  ==  0:
        return 0

    cfg  =  FORMAT_CONFIGS[fmt]
    raw  =  int(np.floor(np.log2(amax)))

    if cfg["kind"]  ==  "mxfp":
        exponent  =  raw  +  cfg["baseline_exp_shift"]
    elif cfg["kind"]  ==  "mxuint":
        negative_extent  =  max(0.0,   - float(np.min(block)))
        positive_extent  =  max(0.0,  float(np.max(block)))
        required_step  =  max(negative_extent  /  8.0,  positive_extent  /  7.0)
        exponent  =  int(np.ceil(np.log2(required_step))) if required_step  >  0 else 0
    elif cfg["kind"]  ==  "bfp":
        if BFP_EXP_POLICY  ==  "autoflex_predict":
            exponent  =  bfp_exp_autoflex_predict(block)
        elif BFP_EXP_POLICY  ==  "autoflex_init":
            exponent  =  bfp_exp_autoflex_init(block)
        else:
            raise ValueError(f"Unknown BFP_EXP_POLICY={BFP_EXP_POLICY}")
    else:
        exponent  =  raw

    return int(np.clip(exponent,  cfg["exp_min"],  cfg["exp_max"]))

def quantize_block_at_exp(block,  fmt,  exponent):
    """Return dequantized values using the brute-force notebook's exact grids."""
    block  =  np.asarray(block,  dtype = np.float64)
    cfg  =  FORMAT_CONFIGS[fmt]
    exponent  =  int(np.clip(exponent,  cfg["exp_min"],  cfg["exp_max"]))
    scale  =  np.exp2(exponent)

    if cfg["kind"]  ==  "mxfp":
        local  =  np.abs(block)  /  scale
        qlocal  =  nearest_grid_rne(local,  cfg["grid"])
        return np.sign(block)  *  qlocal  *  scale

    if cfg["kind"]  ==  "mxint":
        step  =  scale  *  cfg["element_scale"]
        q  =  np.clip(np.rint(block  /  step),  cfg["qmin"],  cfg["qmax"])
        return q  *  step

    if cfg["kind"]  ==  "mxuint":

        q_raw  =  np.rint(block  /  scale)  +  cfg["zero_point"]
        q_unsigned  =  np.clip(q_raw,  cfg["qmin"],  cfg["qmax"])
        q_signed  =  q_unsigned  -  cfg["zero_point"]
        return q_signed  *  scale

    if cfg["kind"]  ==  "bfp":

        mantissa  =  np.clip(np.rint(block  /  scale),  cfg["qmin"],  cfg["qmax"])
        return mantissa  *  scale

    raise ValueError(f"Unknown quantizer kind for {fmt}: {cfg['kind']}")

def quantize_matrix_baseline(W,  fmt):
    """Full matrix baseline quantization; used only by the comparison notebook."""
    cfg  =  FORMAT_CONFIGS[fmt]

    if fmt  ==  "BFP4" and BFP_SHARING  ==  "tensor":
        exp  =  baseline_exp(W.ravel(),  fmt)
        return quantize_block_at_exp(W,  fmt,  exp)

    Wq  =  np.empty_like(W,  dtype = np.float64)

    for r in range(W.shape[0]):
        for start,  end in rowwise_ranges(W.shape[1],  cfg["block_size"]):
            block  =  W[r,  start:end]
            Wq[r,  start:end]  =  quantize_block_at_exp(block,  fmt,  baseline_exp(block,  fmt))

    return Wq

def relative_output_error_pct(W,  Wq,  X):
    Yref  =  W  @  X
    denominator  =  float(np.linalg.norm(Yref))
    numerator  =  float(np.linalg.norm((W  -  Wq)  @  X))

    if denominator  ==  0:
        return 0.0 if numerator  ==  0 else float("inf")

    return 100.0  *  numerator  /  denominator


## Select columns and score formats

This is the main Columnwise-K algorithm. It uses a smaller set of weight columns to estimate which format will work best.

1. **Find important columns.** Take the absolute value of each weight and add the results within its column. A large total means the column contains large weights. The absolute value prevents negative and positive weights from cancelling.
2. **Keep the top columns.** `K_COLUMN_PCT` sets the percentage to keep. Every format uses the same selected columns, so the comparison uses the same data.
3. **Choose scales from original blocks.** Read the complete original block to choose its baseline scale, then quantize only the selected weights inside it. Selecting columns does not regroup weights into new blocks.
4. **Measure selected output error.** Subtract the rounded selected weights from the original selected weights, then multiply by the matching input rows. Add the signed contributions first, square the resulting errors, and sum them. Rank formats by this score; lower is better.

**Example:** suppose three columns are `[1, 1]`, `[4, 3]`, and `[2, 0]`. Their absolute totals are 2, 7, and 2. If we keep one column, we select column 2. Each format rounds that column and uses input row 2 to measure its contribution error.

The algorithm estimates format quality from selected columns. Omitted columns can change the full output error, so the partial-score winner is not guaranteed to be the full-output winner.


In [3]:
FORMATS  =  {}

for _name in FORMAT_ORDER:
    _c  =  dict(FORMAT_CONFIGS[_name])
    _c["name"]  =  _name

    if _c["kind"]  ==  "mxuint":
        _c["midpoint_coded"]  =  True

    FORMATS[_name]  =  _c

MAX_BITS_PER_WEIGHT_UNUSED  =  None

def baseline_exponent(block,  cfg):
    return baseline_exp(block,  cfg["name"])

def quantize_values(values,  cfg,  exponent):
    return quantize_block_at_exp(values,  cfg["name"],  exponent)

def block_ranges(ncols,  block_size):
    return [(s,  min(s  +  block_size,  ncols)) for s in range(0,  ncols,  block_size)]

def storage_for(cfg,  shape):
    n  =  math.prod(shape)
    blocks  =  shape[0]  *  math.ceil(shape[1]  /  cfg["block_size"])
    total  =  n  *  cfg["element_bits"]  +  blocks  *  cfg["metadata_bits"]
    return dict(Blocks = blocks,  Total_Storage_Bits = total,  Bits_per_Weight = total  /  n)

def eligible_formats(W,  formats,  max_bits = None):
    rows,  eligible  =  [],  []

    for name,  cfg in formats.items():
        st  =  storage_for(cfg,  W.shape)
        status,  reason  =  "ELIGIBLE",  "Encoding is defined in the configuration."

        if cfg["kind"]  ==  "mxuint" and not cfg.get("midpoint_coded",  False):
            if np.any(W  <  0):
                status,  reason  =  "INELIGIBLE",  "Negative weights; no auxiliary sign/zero point is permitted here."
            elif not cfg.get("encoding_defined",  False):
                status,  reason  =  "NOT_SCORED",  "Nonnegative weights, but the classroom unsigned encoding is not supplied."

        if status  ==  "ELIGIBLE" and max_bits is not None and st["Bits_per_Weight"]  >  max_bits  +  1e-12:
            status,  reason  =  "OVER_STORAGE_LIMIT",  "Effective storage exceeds the supplied limit."

        if status  ==  "ELIGIBLE":
            eligible.append(name)

        rows.append(dict(Format = name,  Status = status,  Reason = reason,   ** st))

    return eligible,  pd.DataFrame(rows)

def relative_error_pct(sse,  energy):
    if energy  ==  0:
        return 0. if sse  ==  0 else float("inf")

    return 100.  *  math.sqrt(max(0.,  sse)  /  energy)

def choose_columns(W, pct):
    if not np.isfinite(pct) or not 0 < pct <= 100: raise ValueError('K_COLUMN_PCT must be in (0,100].')

    importance = np.sum(np.abs(W), axis = 0)
    order = np.argsort( - importance, kind = 'stable')
    m = int(math.ceil(W.shape[1] * pct / 100))
    ranked = order[:m]

    selected = np.sort(ranked)
    mask = np.zeros(W.shape[1], bool); mask[selected] = True
    table = pd.DataFrame(dict(Importance_rank = np.arange(1, W.shape[1] + 1), Original_column = order, Weight_energy = importance[order], Selected = mask[order]))
    total = float(importance.sum())
    coverage = 100 * float(importance[selected].sum()) / total if total else np.nan
    return selected, table, coverage

def quantize_selected_columns(W, cfg, columns):
    sampled = np.empty((W.shape[0], len(columns)), dtype = np.float64)
    records = []
    count = 0

    for r in range(W.shape[0]):
        for b, (start, end) in enumerate(block_ranges(W.shape[1], cfg['block_size'])):
            positions = np.flatnonzero((columns >= start)&(columns < end))

            if len(positions) == 0: continue

            original = W[r, start:end]
            exponent = baseline_exponent(original, cfg)
            original_ids = columns[positions]
            values = W[r, original_ids]
            sampled[r, positions] = quantize_values(values, cfg, exponent)
            count += len(values)
            records.append(dict(Row = r, Block = b, Start = start, End = end,
                Original_max_abs = float(np.max(np.abs(original))),
                Shared_exponent = exponent, Weights_read_for_scale = end - start,
                Weights_quantized = len(values)))

    assert count == W.shape[0] * len(columns)
    return sampled, count, pd.DataFrame(records)

def select_format(W, X):
    begin = time.perf_counter()
    columns, importance, coverage = choose_columns(W, K_COLUMN_PCT)
    selected_W = W[:, columns]
    selected_X = X[columns, :]
    target = selected_W @ selected_X
    target_energy = float(np.sum(target ** 2))
    eligible, eligibility = eligible_formats(W, FORMATS)

    if not eligible: raise ValueError('No eligible formats.')

    scores = []; row_records = []; trace_records = []

    for fmt in eligible:
        cfg = FORMATS[fmt]
        t = time.perf_counter()
        Wq, count, trace = quantize_selected_columns(W, cfg, columns)
        quant_seconds = time.perf_counter() - t
        t = time.perf_counter()
        error = (selected_W - Wq) @ selected_X
        row_loss = np.sum(error ** 2, axis = 1)
        score_seconds = time.perf_counter() - t
        sse = float(row_loss.sum())
        scores.append(dict(Format = fmt, Sampled_SSE = sse, Sampled_error_pct = relative_error_pct(sse, target_energy),
            Weights_quantized = count, Full_weights = W.size,
            Scoring_terms = count * X.shape[1], Full_scoring_terms = W.size * X.shape[1],
            Reduction_pct = 100 * (1 - count / W.size),
            Scale_scan_values = int(trace.Weights_read_for_scale.sum()),
            Quantize_seconds = quant_seconds, Score_seconds = score_seconds,
            Effective_bits_per_weight = storage_for(cfg, W.shape)['Bits_per_Weight']))
        row_records.extend(dict(Format = fmt, Row = r, Sampled_row_SSE = float(v)) for r, v in enumerate(row_loss))
        trace.insert(0, 'Format', fmt); trace_records.append(trace)

    table = pd.DataFrame(scores).sort_values(['Sampled_SSE', 'Effective_bits_per_weight'], kind = 'stable').reset_index(drop = True)
    table.insert(0, 'Predicted_rank', np.arange(1, len(table) + 1))
    table['Tied_for_best'] = np.isclose(table.Sampled_SSE, table.iloc[0].Sampled_SSE, rtol = 1e-12, atol = 1e-12)
    return dict(columns = columns, importance = importance, energy_coverage_pct = coverage,
        scores = table, row_losses = pd.DataFrame(row_records), block_trace = pd.concat(trace_records, ignore_index = True),
        eligibility = eligibility, winner = table.iloc[0].Format, seconds = time.perf_counter() - begin)

def score_columnwise_k(W,  X,  pct):
    """Thin wrapper: runs the unchanged select_format and renames result columns for the displays below."""
    global K_COLUMN_PCT
    K_COLUMN_PCT  =  pct
    r  =  select_format(W,  X)
    t  =  r["scores"].rename(columns = {
        "Predicted_rank": "ColumnwiseK_rank",
        "Sampled_SSE": "Sampled_contribution_SSE",
        "Sampled_error_pct": "ColumnwiseK_sampled_error_pct",
        "Full_weights": "Original_weight_values",
        "Scoring_terms": "Sampled_scoring_terms",
        "Full_scoring_terms": "Full_scoring_terms_one_format",
        "Reduction_pct": "Quantization_reduction_vs_one_full_pass_pct",
        "Quantize_seconds": "Quantization_seconds",
    })
    t["Scoring_term_reduction_pct"]  =  t["Quantization_reduction_vs_one_full_pass_pct"]
    return {
        "selected_columns": r["columns"],  "importance": r["importance"],
        "energy_coverage_pct": r["energy_coverage_pct"],  "scores": t,
        "row_losses": r["row_losses"],  "block_trace": r["block_trace"],
        "eligibility": r["eligibility"],  "selector_seconds": r["seconds"],
        "selected_column_count": len(r["columns"]),
    }


## Search for better scales

A block's scale controls the spacing between its allowed rounded values. The usual scale is a starting choice; another scale may give a better multiplication result.

For each format and weight row, this cell:

1. Builds nearby exponent choices for each block. An exponent `e` represents a scale of `2^e`.
2. Tries combinations of these choices across the row's blocks.
3. Measures the complete output error against the original multiplication and keeps the best combination.

For example, if two blocks each have three exponent choices, there are `3 × 3 = 9` combinations to check. Checking them together matters because their signed output errors can partly cancel or reinforce each other.

The search normally starts within ±3 of each baseline exponent. If the best choice touches the boundary, it may widen up to ±5. It is exact within the candidates searched, not over every possible exponent. If exact enumeration exceeds the configured size limit, the code raises an error.

The cell also defines error measurements and a function for comparing rankings.


In [4]:
INITIAL_EXP_WINDOW  =  3
MAX_EXP_WINDOW  =  5
MAX_EXACT_SEARCH_ELEMENTS  =  2_000_000
SEARCH_SEED  =  0

def quantization_error_metrics(W,  Wq,  X,  Y_ref = None):
    if Y_ref is None:
        Y_ref  =  W  @  X

    denominator  =  float(np.linalg.norm(Y_ref))
    error_matrix  =  (W  -  Wq)  @  X
    numerator  =  float(np.linalg.norm(error_matrix))
    error_pct  =  0.0 if denominator  ==  0 and numerator  ==  0 else (
        float("inf") if denominator  ==  0 else 100.0  *  numerator  /  denominator
    )
    return {
        "Actual_output_error_pct": error_pct,
        "Output_SSE": float(np.sum(error_matrix  **  2)),
        "Output_SNR_dB": (
            float("inf") if numerator  ==  0 else
            (float("-inf") if denominator  ==  0 else 20.0  *  np.log10(denominator  /  numerator))
        ),
    }

def gram_factor(X):
    """F F^T = X X^T over its numerically non-zero eigenspace."""
    gram  =  X  @  X.T
    eigenvalues,  eigenvectors  =  np.linalg.eigh(gram)

    if eigenvalues.size  ==  0 or float(np.max(eigenvalues))  <=  0:
        return np.zeros((X.shape[0],  0),  dtype = np.float64)

    keep  =  eigenvalues  >  float(np.max(eigenvalues))  *  1e-14
    return eigenvectors[:,  keep]  *  np.sqrt(eigenvalues[keep])

def candidate_exponents(block,  fmt,  window):
    cfg  =  FORMAT_CONFIGS[fmt]
    baseline  =  baseline_exp(block,  fmt)
    offsets  =  sorted(range( - window,  window  +  1),  key = lambda off: (abs(off),  off))
    exponents  =  []

    for off in offsets:
        exponent  =  int(np.clip(baseline  +  off,  cfg["exp_min"],  cfg["exp_max"]))

        if exponent not in exponents:
            exponents.append(exponent)

    return baseline,  np.asarray(exponents,  dtype = int)

def enumerate_exact_block_choices(residual_contributions):
    """Exhaustively combine the candidate residual contribution for each block."""

    if residual_contributions[0].shape[1]  ==  0:

        return [0]  *  len(residual_contributions),  0.0

    combined  =  residual_contributions[0]

    for contribution in residual_contributions[1:]:
        combined  =  (
            combined[:,  None,  :]  +  contribution[None,  :,  :]
        ).reshape( - 1,  combined.shape[ - 1])

    sse  =  np.einsum("ij,ij->i",  combined,  combined)
    best_flat  =  int(np.argmin(sse))
    shape  =  [len(contribution) for contribution in residual_contributions]
    best_indices  =  [int(i) for i in np.unravel_index(best_flat,  shape)]
    return best_indices,  float(sse[best_flat])

def brute_force_alternative_exponents(W,  X,  fmt,  initial_window = INITIAL_EXP_WINDOW,
                                      max_window = MAX_EXP_WINDOW,
                                      max_elements = MAX_EXACT_SEARCH_ELEMENTS):
    """Minimize actual ||(W-Wq)X||_F^2 with exact bounded exponent enumeration.

    Starts at baseline exponent +/- initial_window. If any chosen exponent lies
    on a window edge, the row window is widened up to max_window when the exact
    combination tensor still fits max_elements. No heuristic fallback is used:
    if exact enumeration exceeds the cap, the function stops with an informative error.
    """
    t0  =  time.perf_counter()
    cfg  =  FORMAT_CONFIGS[fmt]
    F  =  gram_factor(X)
    feature_dim  =  max(1,  F.shape[1])
    combination_limit  =  max(1,  int(max_elements // feature_dim))
    ranges  =  rowwise_ranges(W.shape[1],  cfg["block_size"])
    Wq  =  np.empty_like(W,  dtype = np.float64)
    cache  =  {}
    candidate_weight_values  =  0
    candidate_block_quantizations  =  0
    combinations_evaluated  =  0
    max_combinations_per_row  =  0
    widened_rows  =  0
    final_edge_rows  =  0
    chosen_offsets  =  []
    selected_window_per_row  =  []

    for row in range(W.shape[0]):
        row_window  =  initial_window

        while True:
            block_candidates  =  []
            residual_contributions  =  []

            for block_id,  (start,  end) in enumerate(ranges):
                block  =  W[row,  start:end]
                baseline,  exponents  =  candidate_exponents(block,  fmt,  row_window)
                q_options  =  []

                for exponent in exponents:
                    key  =  (row,  block_id,  int(exponent))

                    if key not in cache:
                        cache[key]  =  quantize_block_at_exp(block,  fmt,  int(exponent))
                        candidate_weight_values  +=  block.size
                        candidate_block_quantizations  +=  1

                    q_options.append(cache[key])

                q_options  =  np.asarray(q_options,  dtype = np.float64)
                residual_contributions.append((block[None,  :]  -  q_options)  @  F[start:end])
                block_candidates.append((baseline,  exponents,  q_options))

            combinations  =  math.prod(len(d) for d in residual_contributions)
            max_combinations_per_row  =  max(max_combinations_per_row,  combinations)

            if combinations  *  feature_dim  >  max_elements:
                raise RuntimeError(
                    f"{fmt}, row {row}: {combinations:,} exponent combinations × "
                    f"{feature_dim} feature dimensions exceed the exact-search cap "
                    f"{max_elements:,}. Use the assignment-sized matrix or explicitly "
                    "increase MAX_EXACT_SEARCH_ELEMENTS after checking memory use."
                )

            chosen_indices,  _  =  enumerate_exact_block_choices(residual_contributions)
            combinations_evaluated  +=  combinations
            offsets  =  [
                int(block_candidates[b][1][chosen_indices[b]]  -  block_candidates[b][0])
                for b in range(len(ranges))
            ]
            on_edge  =  any(abs(offset)  ==  row_window for offset in offsets)
            can_widen  =  row_window  <  max_window and (
                (2  *  (row_window  +  1)  +  1)  **  len(ranges)  *  feature_dim  <=  max_elements
            )

            if on_edge and can_widen:
                row_window  +=  1
                continue

            break

        if row_window  >  initial_window:
            widened_rows  +=  1

        if on_edge:
            final_edge_rows  +=  1

        selected_window_per_row.append(row_window)

        for b,  (start,  end) in enumerate(ranges):
            baseline,  exponents,  q_options  =  block_candidates[b]
            selected_exp  =  int(exponents[chosen_indices[b]])
            chosen_offsets.append(selected_exp  -  baseline)
            Wq[row,  start:end]  =  q_options[chosen_indices[b]]

    return Wq,  {
        "Search_method": "EXACT bounded exponent enumeration",
        "Initial_window": initial_window,
        "Maximum_window": max_window,
        "Candidate_block_quantizations": candidate_block_quantizations,
        "Candidate_weight_values_quantized": candidate_weight_values,
        "Exponent_combinations_evaluated": combinations_evaluated,
        "Maximum_combinations_in_one_row_trial": max_combinations_per_row,
        "Rows_widened": widened_rows,
        "Rows_with_final_edge_optimum": final_edge_rows,
        "Minimum_selected_offset": int(min(chosen_offsets)) if chosen_offsets else 0,
        "Maximum_selected_offset": int(max(chosen_offsets)) if chosen_offsets else 0,
        "Nonbaseline_block_offsets": int(sum(int(o  !=  0) for o in chosen_offsets)),
        "Selected_window_min": int(min(selected_window_per_row)) if selected_window_per_row else initial_window,
        "Selected_window_max": int(max(selected_window_per_row)) if selected_window_per_row else initial_window,
        "Search_seconds": time.perf_counter()  -  t0,
    }

def kendall_tau_b_from_ranks(a,  b):
    """Dependency-free Kendall tau-b for two ranking vectors."""
    concordant  =  discordant  =  tie_a  =  tie_b  =  0

    for i in range(len(a)):
        for j in range(i  +  1,  len(a)):
            da  =  np.sign(a[i]  -  a[j])
            db  =  np.sign(b[i]  -  b[j])

            if da  ==  0 and db  ==  0:
                continue

            if da  ==  0:
                tie_a  +=  1
            elif db  ==  0:
                tie_b  +=  1
            elif da  ==  db:
                concordant  +=  1
            else:
                discordant  +=  1

    denominator  =  math.sqrt((concordant  +  discordant  +  tie_a)  *
                            (concordant  +  discordant  +  tie_b))
    return np.nan if denominator  ==  0 else (concordant  -  discordant)  /  denominator


## Run the comparison and read the tables

The next cell runs three methods on the same weights and inputs:

- **Columnwise-K:** rank formats using selected columns at baseline scales.
- **Full baseline:** quantize all weights at baseline scales and measure the complete output error.
- **Alternative-exponent search:** try other block scales and measure the complete output error.

The original multiplication `W @ X_cal` is the reference for the full-output methods. Lower error is better. Columnwise-K uses a selected-column reference, so compare its ranking and chosen format with the other methods rather than directly comparing their error percentages.

The tables show the chosen formats, rankings, storage, work counts, search details, and measured times. A matching winner does not mean the complete rankings match. Reading full blocks for scales still costs work, even when only selected weights are quantized.


In [5]:
W,  X_cal,  INPUT_PROFILE  =  load_inputs()
Y_ref  =  W  @  X_cal
output_energy  =  float(np.sum(Y_ref  **  2))

print("INPUT PROFILE")
display(INPUT_PROFILE)

format_rows  =  []

for fmt in FORMAT_ORDER:
    cfg  =  FORMAT_CONFIGS[fmt]
    baseline_rule  =  {
        "E2M1": "floor(log2(max_abs)) - 2",
        "E1M2": "floor(log2(max_abs)) - 1",
        "MXUINT4": "ceil(log2(max(negative/8, positive/7))); fixed Z=8",
        "MXINT4": "floor(log2(max_abs))",
        "BFP4": "Autoflex predictor: ceil(log2(max_abs)) - 3",
    }[fmt]
    format_rows.append({
        "Format": fmt,
        "Element_bits": cfg["element_bits"],
        "Block_size": cfg["block_size"],
        "Shared_metadata_bits": cfg["metadata_bits"],
        "Baseline_exponent_rule": baseline_rule,
        "Effective_bits_per_weight": storage_for_format(fmt,  W.shape)["Effective_bits_per_weight"],
    })

FORMAT_TABLE  =  pd.DataFrame(format_rows)

COLUMNWISE_RESULT  =  score_columnwise_k(W,  X_cal,  K_COLUMN_PCT)

baseline_results  =  {}
baseline_matrices  =  {}
baseline_elapsed  =  {}
baseline_rows  =  []

for fmt in FORMAT_ORDER:
    start  =  time.perf_counter()
    Wq_base  =  quantize_matrix_baseline(W,  fmt)
    quant_elapsed  =  time.perf_counter()  -  start
    baseline_matrices[fmt]  =  Wq_base

    score_start  =  time.perf_counter()
    metrics  =  quantization_error_metrics(W,  Wq_base,  X_cal,  Y_ref)
    score_elapsed  =  time.perf_counter()  -  score_start
    baseline_results[fmt]  =  metrics
    baseline_elapsed[fmt]  =  {"Quantization_seconds": quant_elapsed,  "Error_evaluation_seconds": score_elapsed}
    baseline_rows.append({
        "Format": fmt,
        "Baseline_actual_output_error_pct": metrics["Actual_output_error_pct"],
        "Baseline_output_SSE": metrics["Output_SSE"],
        "Baseline_output_SNR_dB": metrics["Output_SNR_dB"],
        "Baseline_weights_quantized": W.size,
        "Baseline_quantization_seconds": quant_elapsed,
        "Baseline_error_evaluation_seconds": score_elapsed,
        "Effective_bits_per_weight": storage_for_format(fmt,  W.shape)["Effective_bits_per_weight"],
    })

baseline_table  =  pd.DataFrame(baseline_rows).sort_values(
    ["Baseline_actual_output_error_pct",  "Format"],  kind = "stable"
).reset_index(drop = True)
baseline_table.insert(0,  "Baseline_rank",  np.arange(1,  len(baseline_table)  +  1))

optimized_matrices  =  {}
search_info  =  {}
optimized_results  =  {}
optimized_rows  =  []

for fmt in FORMAT_ORDER:
    Wq_opt,  info  =  brute_force_alternative_exponents(W,  X_cal,  fmt)
    optimized_matrices[fmt]  =  Wq_opt
    search_info[fmt]  =  info
    optimized_results[fmt]  =  quantization_error_metrics(W,  Wq_opt,  X_cal,  Y_ref)
    base_error  =  baseline_results[fmt]["Actual_output_error_pct"]
    opt_error  =  optimized_results[fmt]["Actual_output_error_pct"]
    improvement  =  100.0  *  (base_error  -  opt_error)  /  base_error if base_error  >  0 else 0.0
    optimized_rows.append({
        "Format": fmt,
        "Alternative_exponent_actual_output_error_pct": opt_error,
        "Alternative_exponent_output_SSE": optimized_results[fmt]["Output_SSE"],
        "Alternative_exponent_output_SNR_dB": optimized_results[fmt]["Output_SNR_dB"],
        "Alternative_exponent_improvement_vs_baseline_pct": improvement,
        "Effective_bits_per_weight": storage_for_format(fmt,  W.shape)["Effective_bits_per_weight"],
         ** info,
    })

optimized_table  =  pd.DataFrame(optimized_rows).sort_values(
    ["Alternative_exponent_actual_output_error_pct",  "Format"],  kind = "stable"
).reset_index(drop = True)
optimized_table.insert(0,  "Alternative_exponent_rank",  np.arange(1,  len(optimized_table)  +  1))

cw_table  =  COLUMNWISE_RESULT["scores"].copy()
comparison_matrix  =  (
    cw_table[[
        "Format",  "ColumnwiseK_rank",  "Sampled_contribution_SSE",
        "ColumnwiseK_sampled_error_pct",  "Weights_quantized",
        "Quantization_reduction_vs_one_full_pass_pct",  "Scale_scan_values",
        "Sampled_scoring_terms",  "Quantization_seconds",
    ]]
    .rename(columns = {
        "Weights_quantized": "ColumnwiseK_weights_quantized",
        "Quantization_reduction_vs_one_full_pass_pct": "ColumnwiseK_quantization_reduction_pct",
        "Scale_scan_values": "ColumnwiseK_scale_scan_values",
        "Sampled_scoring_terms": "ColumnwiseK_scoring_terms",
        "Quantization_seconds": "ColumnwiseK_quantization_seconds",
    })
    .merge(
        baseline_table[[
            "Format",  "Baseline_rank",  "Baseline_actual_output_error_pct",
            "Baseline_output_SNR_dB",  "Baseline_weights_quantized",
            "Baseline_quantization_seconds",
        ]],  on = "Format",  how = "left"
    )
    .merge(
        optimized_table[[
            "Format",  "Alternative_exponent_rank",
            "Alternative_exponent_actual_output_error_pct",
            "Alternative_exponent_output_SNR_dB",
            "Alternative_exponent_improvement_vs_baseline_pct",
            "Candidate_block_quantizations",  "Candidate_weight_values_quantized",
            "Exponent_combinations_evaluated",  "Rows_widened",
            "Minimum_selected_offset",  "Maximum_selected_offset",
            "Nonbaseline_block_offsets",  "Search_seconds",
        ]],  on = "Format",  how = "left"
    )
    .merge(
        FORMAT_TABLE[["Format",  "Effective_bits_per_weight"]],  on = "Format",  how = "left"
    )
    .sort_values("ColumnwiseK_rank")
    .reset_index(drop = True)
)

print("FORMAT CONFIGURATION MATRIX")
display(FORMAT_TABLE.round(6))

print("MAIN COMPARISON MATRIX — one row per format; all error metrics are explicitly labelled")
display(comparison_matrix.round(6))

cw_winner_row  =  COLUMNWISE_RESULT["scores"].iloc[0]
baseline_winner_row  =  baseline_table.iloc[0]
optimized_winner_row  =  optimized_table.iloc[0]
cw_winner  =  str(cw_winner_row["Format"])
baseline_winner  =  str(baseline_winner_row["Format"])
optimized_winner  =  str(optimized_winner_row["Format"])
cw_winner_base_error  =  float(baseline_table.loc[baseline_table["Format"]  ==  cw_winner,  "Baseline_actual_output_error_pct"].iloc[0])

winner_summary  =  pd.DataFrame([
    {
        "Method": "Columnwise-K",
        "Selected_format": cw_winner,
        "Selection_objective": "Selected-column calibration contribution (proxy)",
        "Reported_score_or_error_pct": float(cw_winner_row["ColumnwiseK_sampled_error_pct"]),
        "Metric_type": "sampled contribution; not full-matrix output error",
        "Full_output_error_pct_for_same_format_at_baseline_exponents": cw_winner_base_error,
        "Effective_bits_per_weight": float(FORMAT_TABLE.loc[FORMAT_TABLE["Format"]  ==  cw_winner,  "Effective_bits_per_weight"].iloc[0]),
    },
    {
        "Method": "Brute-force baseline",
        "Selected_format": baseline_winner,
        "Selection_objective": "Actual full calibration-output relative error",
        "Reported_score_or_error_pct": float(baseline_winner_row["Baseline_actual_output_error_pct"]),
        "Metric_type": "actual full-matrix output error",
        "Full_output_error_pct_for_same_format_at_baseline_exponents": float(baseline_winner_row["Baseline_actual_output_error_pct"]),
        "Effective_bits_per_weight": float(baseline_winner_row["Effective_bits_per_weight"]),
    },
    {
        "Method": "Brute-force alternative exponent",
        "Selected_format": optimized_winner,
        "Selection_objective": "Actual full calibration-output relative error; bounded exponent search",
        "Reported_score_or_error_pct": float(optimized_winner_row["Alternative_exponent_actual_output_error_pct"]),
        "Metric_type": "actual full-matrix output error",
        "Full_output_error_pct_for_same_format_at_baseline_exponents": float(baseline_table.loc[baseline_table["Format"]  ==  optimized_winner,  "Baseline_actual_output_error_pct"].iloc[0]),
        "Effective_bits_per_weight": float(optimized_winner_row["Effective_bits_per_weight"]),
    },
])

print("WINNER SUMMARY MATRIX")
display(winner_summary.round(6))

selected_columns  =  COLUMNWISE_RESULT["selected_column_count"]
n_formats  =  len(FORMAT_ORDER)
selected_per_format  =  int(W.shape[0]  *  selected_columns)
scale_scans_per_format  =  int(COLUMNWISE_RESULT["scores"]["Scale_scan_values"].sum()  /  n_formats)
selector_score_terms_per_format  =  int(selected_per_format  *  X_cal.shape[1])
full_score_terms_per_format  =  int(W.size  *  X_cal.shape[1])
alt_candidate_total  =  int(sum(search_info[f]["Candidate_weight_values_quantized"] for f in FORMAT_ORDER))
baseline_total  =  int(W.size  *  n_formats)
selector_total  =  int(selected_per_format  *  n_formats)

work_matrix  =  pd.DataFrame([
    {
        "Method_or_stage": "Columnwise-K selection (per format)",
        "Quantizer_input_values_per_format": selected_per_format,
        "Quantizer_input_values_all_5_formats": selector_total,
        "Scoring_terms_per_format": selector_score_terms_per_format,
        "Scoring_terms_all_5_formats": selector_score_terms_per_format  *  n_formats,
        "Full_matrix_weight_values_read_for_column_importance_once": 0,
        "Additional_scale_scan_values_per_format": int(COLUMNWISE_RESULT["scores"]["Scale_scan_values"].iloc[0]),
        "Notes": "Column importance separately examines all W.size values once; quantizes only selected entries",
    },
    {
        "Method_or_stage": "Brute-force baseline (per format)",
        "Quantizer_input_values_per_format": int(W.size),
        "Quantizer_input_values_all_5_formats": baseline_total,
        "Scoring_terms_per_format": full_score_terms_per_format,
        "Scoring_terms_all_5_formats": full_score_terms_per_format  *  n_formats,
        "Full_matrix_weight_values_read_for_column_importance_once": 0,
        "Additional_scale_scan_values_per_format": 0,
        "Notes": "One baseline quantization pass over the complete W for each format",
    },
    {
        "Method_or_stage": "Brute-force alternative-exponent candidate generation",
        "Quantizer_input_values_per_format": np.nan,
        "Quantizer_input_values_all_5_formats": alt_candidate_total,
        "Scoring_terms_per_format": np.nan,
        "Scoring_terms_all_5_formats": np.nan,
        "Full_matrix_weight_values_read_for_column_importance_once": 0,
        "Additional_scale_scan_values_per_format": 0,
        "Notes": "Sum of unique cached block candidates across all formats/windows; varies by format",
    },
])
print("WORK ACCOUNTING MATRIX")
display(work_matrix)

column_importance_work  =  pd.DataFrame([{
    "Operation": "Columnwise-K importance calculation",
    "Weight_values_read": W.size,
    "Scalar_squares": W.size,
    "Selected_columns": selected_columns,
    "Actual_selected_column_pct": 100.0  *  selected_columns  /  W.shape[1],
    "Selected_weight_energy_coverage_pct": COLUMNWISE_RESULT["energy_coverage_pct"],
    "Weights_quantized_by_selector_across_formats": selector_total,
    "Baseline_quantizer_inputs_across_formats": baseline_total,
    "Columnwise_quantizer_input_reduction_pct_vs_baseline": 100.0  *  (1.0  -  selector_total  /  baseline_total),
    "Scoring_term_reduction_pct_vs_full_baseline": 100.0  *  (1.0  -  selector_score_terms_per_format  /  full_score_terms_per_format),
}])
print("COLUMNWISE-K WORK SUMMARY")
display(column_importance_work.round(6))

search_diagnostics  =  optimized_table[[
    "Format",  "Alternative_exponent_rank",  "Search_method",  "Initial_window",  "Maximum_window",
    "Candidate_block_quantizations",  "Candidate_weight_values_quantized",
    "Exponent_combinations_evaluated",  "Maximum_combinations_in_one_row_trial",
    "Rows_widened",  "Rows_with_final_edge_optimum",  "Nonbaseline_block_offsets",
    "Minimum_selected_offset",  "Maximum_selected_offset",  "Search_seconds",
]].copy()
print("ALTERNATIVE-EXPONENT SEARCH DIAGNOSTICS")
display(search_diagnostics.round(6))

format_index  =  {fmt: i for i,  fmt in enumerate(FORMAT_ORDER)}
cw_rank_vec  =  np.array([int(cw_table.loc[cw_table["Format"]  ==  f,  "ColumnwiseK_rank"].iloc[0]) for f in FORMAT_ORDER])
base_rank_vec  =  np.array([int(baseline_table.loc[baseline_table["Format"]  ==  f,  "Baseline_rank"].iloc[0]) for f in FORMAT_ORDER])
opt_rank_vec  =  np.array([int(optimized_table.loc[optimized_table["Format"]  ==  f,  "Alternative_exponent_rank"].iloc[0]) for f in FORMAT_ORDER])
ranking_diagnostics  =  pd.DataFrame([
    {"Comparison": "Columnwise-K ranking vs. brute-force baseline ranking",  "Kendall_tau_b": kendall_tau_b_from_ranks(cw_rank_vec,  base_rank_vec)},
    {"Comparison": "Columnwise-K ranking vs. brute-force alternative-exponent ranking",  "Kendall_tau_b": kendall_tau_b_from_ranks(cw_rank_vec,  opt_rank_vec)},
    {"Comparison": "Brute-force baseline ranking vs. alternative-exponent ranking",  "Kendall_tau_b": kendall_tau_b_from_ranks(base_rank_vec,  opt_rank_vec)},
])
print("RANKING AGREEMENT MATRIX")
display(ranking_diagnostics.round(6))

print("WINNER AGREEMENT MATRIX")
display(pd.DataFrame([{
    "ColumnwiseK_winner": cw_winner,
    "BruteForce_baseline_winner": baseline_winner,
    "BruteForce_alternative_exponent_winner": optimized_winner,
    "ColumnwiseK_matches_baseline": cw_winner  ==  baseline_winner,
    "ColumnwiseK_matches_alternative_exponent": cw_winner  ==  optimized_winner,
    "Baseline_matches_alternative_exponent": baseline_winner  ==  optimized_winner,
}]))

print("TIMING MATRIX")
timing_rows  =  [
    {
        "Stage": "Columnwise-K selector (all five formats)",
        "Seconds": COLUMNWISE_RESULT["selector_seconds"],
        "Timing_scope": "importance, selected-value quantization and sampled scoring",
    },
    {
        "Stage": "Brute-force baseline (all five formats)",
        "Seconds": float(sum(baseline_elapsed[f]["Quantization_seconds"]  +  baseline_elapsed[f]["Error_evaluation_seconds"] for f in FORMAT_ORDER)),
        "Timing_scope": "full-matrix baseline quantization plus actual full-output error evaluation",
    },
    {
        "Stage": "Brute-force alternative-exponent searches (all five formats)",
        "Seconds": float(sum(search_info[f]["Search_seconds"] for f in FORMAT_ORDER)),
        "Timing_scope": "bounded exact exponent search; does not include baseline pass",
    },
]
display(pd.DataFrame(timing_rows).round(6))


INPUT PROFILE


,Dataset,W_shape,X_cal_shape,W_original_dtype,X_cal_original_dtype,W_source,X_cal_source,Preprocessing
0,C,"(100, 100)","(100, 256)",float32,float32,c:/Users/Admin/OneDrive/Desktop/System/W_C.npy,c:/Users/Admin/OneDrive/Desktop/System/X_cal.npy,none (float64 conversion for analysis only)


FORMAT CONFIGURATION MATRIX


,Format,Element_bits,Block_size,Shared_metadata_bits,Baseline_exponent_rule,Effective_bits_per_weight
0,E2M1,4,32,8,floor(log2(max_abs)) - 2,4.32
1,E1M2,4,32,8,floor(log2(max_abs)) - 1,4.32
2,MXUINT4,4,32,8,"ceil(log2(max(negative/8, positive/7))); fixed...",4.32
3,MXINT4,4,32,8,floor(log2(max_abs)),4.32
4,BFP4,4,32,8,Autoflex predictor: ceil(log2(max_abs)) - 3,4.32


MAIN COMPARISON MATRIX — one row per format; all error metrics are explicitly labelled


...

,Comparison,Kendall_tau_b
0,Columnwise-K ranking vs. brute-force baseline ...,0.8
1,Columnwise-K ranking vs. brute-force alternati...,0.6
2,Brute-force baseline ranking vs. alternative-e...,0.8


WINNER AGREEMENT MATRIX


,ColumnwiseK_winner,BruteForce_baseline_winner,BruteForce_alternative_exponent_winner,ColumnwiseK_matches_baseline,ColumnwiseK_matches_alternative_exponent,Baseline_matches_alternative_exponent
0,E2M1,E2M1,E2M1,True,True,True


TIMING MATRIX


,Stage,Seconds,Timing_scope
0,Columnwise-K selector (all five formats),0.563900,"importance, selected-value quantization and sa..."
1,Brute-force baseline (all five formats),1.208045,full-matrix baseline quantization plus actual ...
2,Brute-force alternative-exponent searches (all...,9.180636,bounded exact exponent search; does not includ...


## References

[OCP MX formats](https://www.opencompute.org/documents/ocp-microscaling-formats-mx-v1-0-spec-final-pdf) · [FP4 quantization](https://proceedings.mlr.press/v267/wang25ae.html) · [Flexpoint](https://arxiv.org/abs/1711.02213)

These sources explain the number formats. The selection rules are project methods.
